# Module 4: LangSmith — Prompt Engineering, Observability & Evaluations

> Part of the **Modular Workshops** series. Standalone, ~30 min.

Throughout this module we work with a **crew & catering operations agent** for an airline (introduced just below). Everything — traces, datasets, evals — is scoped to that agent so you leave with a concrete before/after picture of how to observe and improve it.

We cover four parts plus a closing loop:

1. **Prompt engineering** — author, test, and version prompts in the Playground and Prompt Hub, then pull them into code with the SDK.
2. **Tracing** — generate traces with the crew & catering agent, then query them with `list_runs` + filters.
3. **Offline evaluations** — build a dataset, score the agent end-to-end (final-response with LLM-as-judge) and step-by-step (trajectory).
4. **Online evaluations** — score every new trace as it lands. Programmatic version + UI workflow.

Then **annotation queues** close the loop: route runs flagged by eval scores to a human for review.

<img src="../images/evals-conceptual.png" style="width: auto; max-height: 400px; border-radius: 8px;">

## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.models import model
from utils.langsmith_rules import (
    get_or_create_annotation_queue,
    create_run_rule,
    delete_run_rule,
)

import os, time
from datetime import datetime, timedelta, timezone
from typing_extensions import TypedDict
from langchain_core.messages import SystemMessage, HumanMessage
from langsmith import Client, uuid7

client = Client()

# Every org-level asset this notebook creates (prompts, datasets, the context
# repo, annotation queues, run rules, experiments) is namespaced to the workshop
# user so attendees sharing one LangSmith workspace don't collide. Tracing
# projects are already per-user via LANGSMITH_PROJECT.
import re
WORKSHOP_USER = os.environ.get("WORKSHOP_USER", "").strip()
# Slugify for use in names/handles: lowercase, hyphen-separated, alphanumerics only.
_user_slug = re.sub(r"[^a-z0-9]+", "-", WORKSHOP_USER.lower()).strip("-") or "shared"

def qualify(name: str) -> str:
    """Suffix a LangSmith asset name with the workshop user's slug."""
    return f"{name}-{_user_slug}"

print("LANGSMITH_TRACING:", os.environ.get("LANGSMITH_TRACING", "not set"))
print("Project:", os.environ.get("LANGSMITH_PROJECT", "default"))
print("Workshop user:", WORKSHOP_USER or "(not set -> using 'shared')")


## The Agent: Crew & Catering Operations

The agent we observe and improve in this module is a small **Deep Agent** for an airline's crew and catering desk. It handles three jobs:

- **Crew scheduling & compliance** — look up who's rostered on a flight and check planned duty periods against duty-time / rest limits.
- **Catering & special meals** — pull the special-meal (IATA-coded) provisioning plan for a flight.
- **Disruption recovery** — propose a crew recovery plan when a flight is delayed or cancelled, re-checking duty limits before re-assignment.

It's a supervisor + a `crew-ops-agent` subagent, wired to domain tools — `crew_roster`, `duty_time_check`, `catering_plan`, `rebook_crew` — plus a `policy_lookup` web-search tool for regulatory/dietary questions.

Rather than treat it as a finished black box, we'll **compose it in three pieces** so each part is inspectable and versioned:

1. **Tools** — the domain functions above (defined in `agents/research_agent.py`).
2. **Memory** — an `AGENTS.md` operating manual: the agent's workflow and rules.
3. **Skills** — `crew-briefing` and `catering-summary` output formats it can reach for.

We'll put the manual and skills in a **LangSmith Context Hub** repo — versioned in the Hub, pulled at build time — instead of pinning them to a local checkout. The next few cells walk through it. (We keep the agent lightweight — no HITL, no disk writes — so evaluation runs don't pause or leak files.)


### Compose it: inspect the manual and skills

The operating manual and skill files live under `agents/deep_agent/` — the same source of truth the deployable agent uses. Let's read them before we ship them to the Hub.


In [ ]:
from pathlib import Path

# The agent's memory + skills live alongside the deployable agent.
assets_dir = project_root / "agents" / "deep_agent"

# Memory: the operating manual.
agents_md = (assets_dir / "AGENTS.md").read_text()

# Skills: each is a folder with a SKILL.md (YAML frontmatter + instructions).
skill_files = {
    f"skills/{p.parent.name}/SKILL.md": p.read_text()
    for p in sorted(assets_dir.glob("skills/*/SKILL.md"))
}

print("AGENTS.md (first 12 lines):")
print("\n".join(agents_md.splitlines()[:12]))
print("\nSkills found:", list(skill_files))


### Compose it: push the manual + skills to Context Hub

**Context Hub** stores an agent's files (manual, skills, notes) in a versioned Hub repo, so the agent can pull them at build time instead of reading a local checkout. We push `AGENTS.md` and the skill files as one repo; each `push_agent` call is a new commit, so the manual and skills get the same version history as prompts do.


In [ ]:
from langsmith.schemas import FileEntry

context_repo = f"-/{qualify('crew-catering-context')}"   # "-" = your default workspace tenant; per-user repo

# One file entry per path: the manual plus every skill's SKILL.md.
files = {"AGENTS.md": FileEntry(content=agents_md)}
files.update({path: FileEntry(content=body) for path, body in skill_files.items()})

commit_url = client.push_agent(
    context_repo,
    files=files,
    description="Crew & catering operations manual + skills",
)
print("Context Hub commit (click to open):", commit_url)


### Compose it: build the agent from Context Hub

Now build the agent with `context_repo=` set. The factory mounts that repo at `/context/` and reads its memory from `/context/AGENTS.md` and its skills from `/context/skills/` — so the manual and skills come from the Hub, not the local disk. We build once and reuse this `agent` for the warm-up and every eval below.


In [ ]:
from agents.research_agent import build_crew_catering_agent

# Build once against the Context Hub repo we just pushed.
agent = build_crew_catering_agent(context_repo=context_repo)

# Peek at the structure: tools + subagent, plus the skills & memory middleware
# that the manual and skills add to the graph.
print("Graph nodes:", list(agent.get_graph().nodes))

quick = agent.invoke(
    {"messages": [{"role": "user", "content": "Who is the crew on flight AC418?"}]},
    config={"configurable": {"thread_id": str(uuid7())}},
)
print("\nSample answer:\n" + quick["messages"][-1].text)


### The shape of the agent

Before we observe or evaluate it, let's *see* the agent: its graph and the tools it can call. 
The supervisor delegates to a `crew-ops-agent` subagent via the `task` tool, and the domain 
tools (`crew_roster`, `duty_time_check`, `catering_plan`, `rebook_crew`, `policy_lookup`) run 
inside the tool node. The Mermaid diagram below is rendered live from the compiled graph.


In [ ]:
from IPython.display import Image, display

# Render the compiled agent graph. draw_mermaid_png() calls a remote renderer,
# so fall back to the Mermaid text diagram if that isn't reachable.
graph = agent.get_graph()
try:
    display(Image(graph.draw_mermaid_png()))
except Exception as e:
    print(f"(PNG render unavailable: {e})\n")
    print(graph.draw_mermaid())

# Inventory the tools the agent can call, so teams see its surface area.
from agents.research_agent import (
    crew_roster, duty_time_check, catering_plan, rebook_crew, policy_lookup,
)
print("\nTools available to the agent:")
for tool_fn in [crew_roster, duty_time_check, catering_plan, rebook_crew, policy_lookup]:
    print(f"  - {tool_fn.name}: {tool_fn.description.splitlines()[0]}")


## Compare two models: efficacy vs. cost

Before diving into LangSmith's features, a practical first question for any team shipping an 
agent: **which model should power it?** A larger model is usually more capable but slower and 
pricier; a smaller model is cheaper and faster but may cut corners. LangSmith experiments make 
this an evidence-based decision instead of a guess.

We run the **same crew & catering agent** over the **same tasks** with two different-sized 
models, then compare:

- **Efficacy** — a strict `correctness` score from an LLM-as-judge (did the agent nail *every* part?).
- **Cost & speed** — total tokens and wall-clock latency per run, pulled back from the traces.

The tasks are deliberately **multi-step** — chain several tool calls, then reason or do arithmetic 
over the results (check every crew member, combine a recovery plan with meal counts, compare two 
flights, catch a boundary compliance case). That's exactly where a smaller model tends to skip a 
step or miscompute, so the strict judge separates the two models instead of scoring both 100%.

We compare **`gpt-5.6-terra`** (larger/default) against **`gpt-5.4-mini`** (smaller/cheaper). 
Each run is a LangSmith experiment, so you can open both in the UI and diff them side by side.


In [ ]:
from langchain.chat_models import init_chat_model
from agents.research_agent import build_crew_catering_agent

# The two models to compare: larger/default vs. smaller/cheaper.
# Both route through the LangSmith Gateway like the rest of the workshop.
MODELS_TO_COMPARE = {
    "gpt-5.6-terra": "large (default)",
    "gpt-5.4-mini": "small (cheaper)",
}

def make_model(model_name):
    return init_chat_model(
        model=model_name,
        model_provider="openai",
        base_url="https://gateway.smith.langchain.com/openai",
        use_responses_api=True,
        api_key=os.environ["LANGSMITH_API_KEY_GATEWAY"],
    )

# Tasks chosen to SEPARATE the models: each needs several chained tool calls
# plus reasoning/arithmetic over the results, where a smaller model tends to
# skip a step, miscompute, or stop early. `required` lists the concrete facts a
# complete, correct answer must contain -- the judge checks every one.
cmp_examples = [
    {
        "inputs": {"query": (
            "For flight AC418, check whether EACH rostered crew member's planned "
            "duty period is compliant if the flight-duty period runs 13.5 hours. "
            "Call out anyone who is non-compliant and by how much."
        )},
        "outputs": {"reference_answer": (
            "Must evaluate ALL four AC418 crew at 13.5h FDP against their limits (the tool "
            "gives flight crew AND the Purser a 13h limit, and only Cabin Crew a 14h limit): "
            "the Captain, First Officer, and Purser (13h limit) are each NON-COMPLIANT by 0.5h; "
            "the Cabin Crew (14h limit) is COMPLIANT with 0.5h margin. "
            "A correct answer names all four roles and the correct verdict + margin/overage for each."
        )},
    },
    {
        "inputs": {"query": (
            "AC902 is cancelled. Give a crew recovery plan, AND confirm the special-meal "
            "count that must be re-timed with catering, AND state how many crew are affected."
        )},
        "outputs": {"reference_answer": (
            "Must combine THREE things for AC902: (1) a recovery plan that holds the assigned "
            "crew and re-checks duty limits before re-assignment, (2) the correct affected-crew "
            "count of 3, and (3) the total special meals to re-time = 8 (VGML 2 + DBML 1 + MOML 5). "
            "All three must be present and the numbers correct."
        )},
    },
    {
        "inputs": {"query": (
            "Compare flights AC418 and AC902: which one has more total special meals loaded, "
            "and how many more? Show the totals for both."
        )},
        "outputs": {"reference_answer": (
            "Must look up BOTH flights and give both totals: AC418 = 15 (VGML 6 + GFML 3 + "
            "KSML 2 + CHML 4) and AC902 = 8 (VGML 2 + DBML 1 + MOML 5), then conclude AC418 "
            "has 7 more. Both totals AND the difference must be correct."
        )},
    },
    {
        "inputs": {"query": (
            "For flight AC902, list the crew, then determine whether the Captain is compliant "
            "at a 13h duty period and whether the Purser is compliant at a 14h duty period."
        )},
        "outputs": {"reference_answer": (
            "Must list the AC902 crew (Captain, First Officer, Purser) AND give two correct "
            "verdicts: the Captain at 13.0h is COMPLIANT (exactly at the 13h limit, 0h margin), "
            "but the Purser at 14.0h is NON-COMPLIANT by 1.0h (the tool gives the Purser a 13h "
            "limit, not 14h). Both verdicts must be correct -- the Purser is the trap."
        )},
    },
]

cmp_dataset_name = qualify("model-comparison-evals")
if client.has_dataset(dataset_name=cmp_dataset_name):
    client.delete_dataset(dataset_id=client.read_dataset(dataset_name=cmp_dataset_name).id)
cmp_dataset = client.create_dataset(cmp_dataset_name)
client.create_examples(
    inputs=[e["inputs"] for e in cmp_examples],
    outputs=[e["outputs"] for e in cmp_examples],
    dataset_id=cmp_dataset.id,
)
print(f"Comparison dataset '{cmp_dataset_name}' ready with {len(cmp_examples)} examples.")


In [ ]:
# A STRICT correctness judge for the comparison. Unlike a lenient 'good enough'
# grader, this one fails the response if ANY required element is missing or any
# number is wrong -- which is what surfaces a smaller model skipping a step or
# miscomputing. A small model keeps judging cheap and consistent across runs.
class CmpGrade(TypedDict):
    """Strict score: does the response contain every required element, all correct?"""
    score: bool
    reasoning: str

cmp_judge_prompt = (
    "You are a STRICT grader for a crew & catering operations assistant.\n"
    "The success rubric lists the specific elements a complete, correct answer must "
    "contain (roles, verdicts, counts, totals, differences).\n\n"
    "Mark score=True ONLY IF the response includes EVERY required element AND every "
    "number/verdict is correct. Mark score=False if the response omits any required "
    "element, gets any number or compliance verdict wrong, stops early, or only "
    "partially answers a multi-part request. Do not give credit for being 'close.'\n"
    "In one sentence, state exactly which required elements are present/correct and "
    "which are missing/wrong."
)
cmp_judge = make_model("gpt-5.4-mini").with_structured_output(CmpGrade)

def cmp_correctness(inputs, outputs, reference_outputs):
    grade = cmp_judge.invoke([
        SystemMessage(content=cmp_judge_prompt),
        HumanMessage(content=(
            f"User request: {inputs['query']}\n\n"
            f"Assistant response: {outputs['response']}\n\n"
            f"Success rubric (required elements): {reference_outputs['reference_answer']}"
        )),
    ])
    return {"key": "correctness", "score": int(grade["score"]), "comment": grade["reasoning"]}


In [ ]:
# Run one experiment per model. The target builds the agent on that model,
# and each row keeps the run (latency + token cost) and the correctness score.
cmp_results = {}

for model_name, label in MODELS_TO_COMPARE.items():
    cmp_agent = build_crew_catering_agent(model=make_model(model_name))

    def run_on_model(inputs, _agent=cmp_agent):
        result = _agent.invoke(
            {"messages": [{"role": "user", "content": inputs["query"]}]},
            config={"configurable": {"thread_id": str(uuid7())}},
        )
        return {"response": result["messages"][-1].text}

    cmp_results[model_name] = client.evaluate(
        run_on_model,
        data=cmp_dataset_name,
        evaluators=[cmp_correctness],
        experiment_prefix=qualify(f"model-cmp-{model_name}"),
        metadata={"model": model_name, "size": label},
        max_concurrency=2,
    )
    print(f"{model_name} ({label}) -> {cmp_results[model_name].experiment_name}")


In [ ]:
import time as _time
from statistics import mean

# Summarize each experiment: efficacy (correctness) + speed (latency) come
# straight from the result rows; cost (tokens) is read back from the traces,
# because token usage is aggregated server-side and isn't on the local run
# object. Tokens land a few seconds after the run, so we retry briefly.
def _tokens_by_experiment(experiment_name, retries=6, delay=5):
    for _ in range(retries):
        runs = list(client.list_runs(project_name=experiment_name, is_root=True))
        toks = [r.total_tokens for r in runs if r.total_tokens]
        if toks:
            return mean(toks)
        _time.sleep(delay)
    return None

def summarize(results):
    scores, latencies = [], []
    for row in results:
        for r in row["evaluation_results"]["results"]:
            if r.key == "correctness" and r.score is not None:
                scores.append(r.score)
        # RunTree exposes .latency (seconds); it has no token fields.
        if row["run"].latency is not None:
            latencies.append(row["run"].latency)
    return {
        "correctness": mean(scores) if scores else None,
        "avg_latency_s": mean(latencies) if latencies else None,
        "avg_tokens": _tokens_by_experiment(results.experiment_name),
    }

print(f"{'model':<16}{'size':<16}{'correctness':>12}{'avg latency':>14}{'avg tokens':>13}")
print("-" * 71)
for model_name, label in MODELS_TO_COMPARE.items():
    s = summarize(cmp_results[model_name])
    corr = f"{s['correctness']:.0%}" if s['correctness'] is not None else "n/a"
    lat = f"{s['avg_latency_s']:.1f}s" if s['avg_latency_s'] is not None else "n/a"
    tok = f"{s['avg_tokens']:.0f}" if s['avg_tokens'] is not None else "n/a"
    print(f"{model_name:<16}{label:<16}{corr:>12}{lat:>14}{tok:>13}")

print("\nTip: open both experiments in LangSmith and use the Compare view to diff")
print("per-example correctness, latency, and cost side by side.")


**Reading the result.** On these multi-step tasks you should see the models **separate** — the 
larger model typically scores higher `correctness` because it completes every required element and 
gets the boundary compliance cases right, while the smaller model more often drops a crew member, 
miscounts meals, or misses the Purser's 13h limit. The table quantifies the trade: how much accuracy 
you'd give up to save on latency and tokens. If the smaller model *does* keep pace here, that's a 
strong signal it's good enough for this workload. Either way you now have evidence, and the rest of 
this module gives you the tools (tracing, datasets, judges, online evals) to keep making that call 
as the agent evolves.


## Part 1. Prompt Engineering — Playground & Prompt Hub

Before you can observe or evaluate an agent, you need a prompt worth shipping. LangSmith treats prompts as **versioned artifacts** — author and test them in the **Playground**, version and share them in the **Prompt Hub**, then pull them into code with the SDK. Three surfaces, one source of truth.

- **Playground** (UI) — an interactive editor: compose messages, wire up input variables, pick a model, and run.
- **Prompt Hub** (UI) — every saved prompt with full commit history, tags, and a public hub of community prompts to fork.
- **SDK** — `push_prompt` / `pull_prompt` to move prompts between code and the hub.

### 1.1 The Prompt Playground

Open **Prompts** in the LangSmith sidebar and click **+ Prompt** to land in the Playground. The left panel is your prompt — an ordered list of messages, each with a role:

- **System** — the instruction manual: persona and ground rules.
- **Human** — the user's turn.
- **AI** — a model turn, handy for few-shot examples.
- **Tool** — tool output, for testing how the model reacts to it.

Add an input variable by typing `{variable_name}` into any message (or highlight text and click **Convert to variable**). Fill in sample values in the right panel's **Inputs** box, then click **Start** to run and see the response.

**Template format.** Variables default to Python **f-string** syntax (`{topic}`). Switch to **mustache** (`{{topic}}`) from the format dropdown when you need loops, conditionals, or nested data (`{{user.name}}`) — f-strings only do flat substitution.

**Model configuration.** Click the **gear icon** next to the model name to set provider, model, temperature, and max tokens. Hit **Save As** to name a configuration — it's shared across your workspace and reusable in other LangSmith features.

**Tools.** Click **+ Tool** to attach tools: built-in ones (web search, code interpreter) or custom tools you define with a name, description, and argument schema. When the model calls a tool, the Playground shows the tool name and arguments so you can verify the call.

🔗 **Try it:** [Open Prompts in LangSmith →](https://smith.langchain.com/prompts) — then click **+ Prompt** (top right) to open the Playground.

### 1.2 Prompt Hub — save, version, share

Click **Save** in the Playground and your prompt lands in the **Prompts** table. Each prompt gets its own detail page with a two-pane layout: commit history and environments on the left, the selected commit on the right.

- **Commits** — every save is a new commit, and the full history is preserved. Toggle **Diff** (top-right) to compare a commit with its predecessor.
- **Tags** — mark a commit with a stable name (e.g. `prod`) so code can reference it without pinning a hash. Move or delete tags as the prompt evolves.
- **Environments** — reserved **Staging** and **Production** environments track which commit is live; **Promote** a commit to move it forward, or roll back from history.
- **Public hub** — search community prompts by name, use case, or model, and **fork** any of them into your workspace.

🔗 **Open in LangSmith:** [Your prompts →](https://smith.langchain.com/prompts) · [Public LangChain Hub →](https://smith.langchain.com/hub)

### 1.3 Manage prompts programmatically

Anything you do in the UI you can do from the SDK: `push_prompt` sends a prompt to the hub, and `pull_prompt` fetches it back. We'll do it in three quick steps — **push** a crew-briefing prompt, **pull it and run it as an agent** (with `create_agent`, not a raw chain), then **version** it.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Step 1 — author the assistant's system prompt and push it to the hub.
prompt_name = qualify("crew-briefing-assistant")
prompt = ChatPromptTemplate([
    ("system",
     "You are a crew operations briefing assistant for an airline. "
     "Look up the flight, then produce a concise pre-departure crew briefing: "
     "(1) a one-line summary, (2) crew on duty and any duty-time/rest risks, "
     "(3) three catering/special-meal notes, and (4) two questions ops should confirm. "
     "Be factual and operational; flag any compliance risk explicitly."),
])

url = client.push_prompt(prompt_name, object=prompt)
print("Prompt page (click to open):", url)

**Pull it back and run it — as an agent, not a chain.** `pull_prompt` returns the prompt we just pushed; we use it as the system prompt for a **separate** `create_agent` demo (named `briefing_agent` so it doesn't disturb the composed agent above), running on the workshop's shared `model`. A small mock `lookup_flight` tool lets the agent fetch the flight's crew and catering details, so the full agent loop (model → tool → model) shows up in the trace.

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def lookup_flight(flight_no: str) -> str:
    """Look up a flight's assigned crew and catering plan by flight number."""
    # Mock ops lookup — swap for your real crew/catering systems in production.
    directory = {
        "AC418": (
            "AC418 (dawn departure). Crew: Capt. Diane Okafor, F/O Marco Ruiz, "
            "Purser Priya Chandran, 1 cabin crew. Planned flight-duty period 12.5h "
            "(within limits). Catering: 6 VGML, 3 GFML, 2 KSML, 4 CHML special meals."
        ),
    }
    return directory.get(flight_no.upper(), f"No flight record found for {flight_no!r}.")


# Step 2 — pull the prompt back and run it with create_agent (uses the imported `model`).
pulled = client.pull_prompt(prompt_name)
system_prompt = pulled.format_messages()[0].content

# Name this demo agent `briefing_agent` so it doesn't clobber the composed
# crew & catering `agent` we built above (the warm-up and evals reuse that one).
briefing_agent = create_agent(model=model, tools=[lookup_flight], system_prompt=system_prompt)

result = briefing_agent.invoke(
    {"messages": [{"role": "user", "content": "Brief me on flight AC418 before departure."}]}
)
print(result["messages"][-1].text)

**Version it.** Re-push under the same name and LangSmith records a new commit — the earlier version stays in the history.

In [ ]:
# Step 3 — re-push a tweaked version. Same name -> a new commit (full history preserved).
prompt_v2 = ChatPromptTemplate([
    ("system",
     "You are a crew operations briefing assistant for an airline. "
     "Look up the flight, then produce a concise pre-departure crew briefing: "
     "(1) a one-line summary, (2) crew on duty and any duty-time/rest risks, "
     "(3) three catering/special-meal notes, (4) two questions ops should confirm, "
     "and (5) one compliance flag to keep in mind. "
     "Be factual and operational; flag any compliance risk explicitly."),
])
url_v2 = client.push_prompt(prompt_name, object=prompt_v2)
print("New commit (click to open):", url_v2)

# Pull a specific commit with client.pull_prompt(f"{prompt_name}:<commit-hash>"),
# and tear down with client.delete_prompt(prompt_name) when you're done.

## Warm-up: Generate a few traces

Before we look at tracing and querying, let's actually produce some traces. 
We reuse the composed `agent` from above and invoke it three times with **deliberately light** prompts — 
each one is a single, focused ops question so the runs finish in a few seconds instead of a few minutes.

On the trial run while building this module the warm-up took **~9 seconds total (3.1s avg per call)**. Expect similar.


In [ ]:
# Reuse the `agent` built above.
warmup_prompts = [
    "Who is the crew on flight AC418? One line per crew member.",
    "Is a 12.5h flight-duty period compliant for a Captain? One sentence.",
    "How many special meals are loaded on flight AC418? One sentence.",
]

total = 0.0
for q in warmup_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].text[:120]}")

print(f"\nTotal: {total:.1f}s ({total/len(warmup_prompts):.1f}s avg)")


## Part 2. Tracing + Querying Traces

Set `LANGSMITH_TRACING=true` and every LLM call, tool call, and state transition lands in your tracing project — no code changes required. 
(The warm-up above already produced traces; this section pulls them back out.)

We use `client.list_runs(...)` to query them.

In [ ]:
project_name = os.environ.get("LANGSMITH_PROJECT", "modular-workshops")
try:
    project = client.read_project(project_name=project_name)
    print(f"Project: {project.name}")
    print(f"View traces: {project.url}")
except Exception as e:
    print(f"Could not read project (this is fine if first run): {e}")


### 2.1 Pull recent traces

Useful filters on `client.list_runs(...)`:

- `project_name=` — scope to one project
- `start_time=` / `end_time=` — time window
- `run_type=` — `"llm"`, `"tool"`, `"chain"`, `"retriever"`
- `error=True` — only failed runs
- `is_root=True` — only top-level traces (not their children)
- `filter=` — LangSmith filter DSL (latency, feedback, attributes...)

In [ ]:
from datetime import datetime, timedelta, timezone

# Pull the last hour of root traces from this workshop's project
since = datetime.now(timezone.utc) - timedelta(hours=1)

recent_runs = list(client.list_runs(
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    start_time=since,
    is_root=True,
    limit=20,
))

print(f"Found {len(recent_runs)} root run(s) in the last hour\n")
for r in recent_runs[:5]:
    latency = (r.end_time - r.start_time).total_seconds() if r.end_time else None
    print(f"- {r.id}  {r.name:25s}  latency={latency}s  error={r.error is not None}")


### 2.2 Filter DSL — find slow or errored runs

The `filter` argument is a small expression language. Common patterns:

- `gt(latency, 5)` — slower than 5 seconds
- `eq(status, "error")` — failed runs
- `and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))` — low-scored runs on a feedback key
- Combine with `and(...)` / `or(...)`

In [ ]:
# Find slow root runs in the last hour (>5s latency)
slow_runs = list(client.list_runs(
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    start_time=since,
    is_root=True,
    filter='gt(latency, 5)',
    limit=20,
))

print(f"{len(slow_runs)} slow root run(s) (>5s) in the last hour")
for r in slow_runs[:5]:
    latency = (r.end_time - r.start_time).total_seconds()
    print(f"  {r.name:25s}  {latency:.1f}s  {r.id}")


## Part 3. Offline Evaluations

**Offline evals** are the experiments you run on demand against a fixed dataset. 
Build a dataset once, score your agent against it whenever you change a prompt, a model, or a tool — get a clean before/after comparison.

Three pieces:
1. **Dataset** — labeled `inputs` + expected `outputs`
2. **Target function** — runs your agent on each example
3. **Evaluators** — score the output (LLM-as-judge or code-based)

### 3.1 Dataset

Same input set, two reference shapes — one for final-response judging, one for trajectory matching. Each example is a real crew/catering ops request; the `trajectory` lists the tool calls we expect (the supervisor delegates to its subagent with `task`, which then calls the domain tools).

In [ ]:
examples = [
    {
        "inputs": {"query": "Who is the crew rostered on flight AC418?"},
        "outputs": {
            "reference_answer": "The captain, first officer, purser, and cabin crew assigned to AC418",
            "trajectory": ["task"],
        },
    },
    {
        "inputs": {"query": "Is a 14.5h flight-duty period compliant for a Captain on AC902?"},
        "outputs": {
            "reference_answer": "It is NON-COMPLIANT — 14.5h exceeds the 13h duty limit for a Captain",
            "trajectory": ["task"],
        },
    },
    {
        "inputs": {"query": "What special meals are loaded on flight AC418, and how many in total?"},
        "outputs": {
            "reference_answer": "The IATA special-meal counts for AC418 and their total",
            "trajectory": ["task"],
        },
    },
    {
        "inputs": {"query": "AC902 is delayed 3 hours. First check the crew roster, then give a recovery plan."},
        "outputs": {
            "reference_answer": "The AC902 roster followed by a crew recovery plan that re-checks duty limits",
            "trajectory": ["task", "task"],
        },
    },
]

dataset_name = qualify("modular-workshops-evals")

if client.has_dataset(dataset_name=dataset_name):
    existing = client.read_dataset(dataset_name=dataset_name)
    client.delete_dataset(dataset_id=existing.id)
    print(f"Deleted existing dataset '{dataset_name}'")

dataset = client.create_dataset(dataset_name)
client.create_examples(
    inputs=[e["inputs"] for e in examples],
    outputs=[e["outputs"] for e in examples],
    dataset_id=dataset.id,
)
print(f"Created dataset '{dataset_name}' with {len(examples)} examples")
print(f"View: {dataset.url}")


### 3.2 Final-response eval (LLM-as-judge)

<img src="../images/final-response.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Treat the agent as a black box: did the final response satisfy the request? 
We'll build the **LLM-as-judge from scratch** so the moving parts are clear:

1. A Pydantic / TypedDict schema for the judge's output (`score`, `reasoning`)
2. A judge prompt that explains the grading criteria
3. `model.with_structured_output(...)` to force the LLM into the schema
4. An evaluator function that calls the judge and returns the score in the shape `client.evaluate` expects

In [ ]:
def run_agent_final(inputs: dict) -> dict:
    """Run the agent and return its response plus any files it wrote.

    The evaluator receives these as separate fields so it can inspect both
    the assistant's message and the resulting file artifacts.
    """
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )

    return {
        "response": result["messages"][-1].text,
        "files": result.get("files") or {},
    }

In [ ]:
from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

# Define the judge's output schema -- with_structured_output enforces this shape on the LLM response.
class CorrectnessGrade(TypedDict):
    """Score whether the agent's response satisfied the user's request."""
    score: bool   # True if correct/helpful, False otherwise
    reasoning: str  # one-sentence explanation

# The dataset's `reference_answer` is a SUCCESS RUBRIC, not an expected response text.
# Make that explicit to the judge so it doesn't downscore valid agent responses that
# happen to be worded differently.
correctness_judge_prompt = """You are an expert grader evaluating an AI assistant.

You will see:
1. The user's request
2. The assistant's final response
3. Any files written by the assistant
4. A success rubric

Evaluate both the response and the files. If the task asks the assistant to write a file, inspect the file contents when available. Mark `score=True` if the task was completed successfully, even if the wording differs from the rubric.

Mark `score=False` only if the task was clearly missed, the file is missing or incorrect, the response contains errors, or the assistant refused without good reason.

Give one short sentence of reasoning.
"""
judge_model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    base_url="https://gateway.smith.langchain.com/openai",
    api_key=os.environ["LANGSMITH_API_KEY_GATEWAY"],
    temperature=0
)
# Bind the schema once -- `judge` is now a structured-output LLM.
judge = model.with_structured_output(CorrectnessGrade)

def correctness_evaluator(inputs, outputs, reference_outputs):
    grade = judge.invoke([
        SystemMessage(content=correctness_judge_prompt),
        HumanMessage(content=(
            f"User request: {inputs['query']}\n\n"
            f"Assistant response: {outputs['response']}\n\n"
            f"Files written: {outputs.get('files', {})}\n\n"
            f"Success rubric: {reference_outputs['reference_answer']}"
        )),
    ])

    return {
        "key": "correctness",
        "score": int(grade["score"]),
        "comment": grade["reasoning"],
    }

In [ ]:
results = client.evaluate(
    run_agent_final,
    data=dataset_name,
    evaluators=[correctness_evaluator],
    experiment_prefix=qualify("final-response"),
    max_concurrency=2,
)
print(f"View at: {results.experiment_name}")


### 3.3 Trajectory eval

<img src="../images/trajectory.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Score the **sequence of tool calls** the agent took, not just the final answer. Three evaluators:

- **`exact_match`** — did it take exactly the right steps in order?
- **`extra_steps`** — how many extra tool calls did it make?
- **`missing_steps`** — how many expected steps did it skip?

`extra_steps` and `missing_steps` use `collections.Counter` for multiset diffs — order doesn't matter for those two, but `exact_match` still catches ordering bugs.

In [ ]:
from collections import Counter
from typing import Any

def trajectory_match(outputs, reference_outputs):
    return {
        "key": "exact_match",
        "score": int(outputs["trajectory"] == reference_outputs["trajectory"]),
    }

def extra_steps(outputs, reference_outputs):
    extras = Counter(outputs["trajectory"]) - Counter(reference_outputs["trajectory"])
    return {"key": "extra_steps", "score": sum(extras.values())}

def missing_steps(outputs, reference_outputs):
    missing = Counter(reference_outputs["trajectory"]) - Counter(outputs["trajectory"])
    return {"key": "missing_steps", "score": sum(missing.values())}


Next, we'll define the run function.

In [ ]:
def extract_tool_calls(messages: list[Any]) -> list[str]:
    """Extract tool call names from messages in order."""
    tool_names = []
    for msg in messages:
        if getattr(msg, "tool_calls", None):
            tool_names.extend(tc["name"] for tc in msg.tool_calls)
    return tool_names

def run_agent_trajectory(inputs: dict) -> dict:
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )
    return {"trajectory": extract_tool_calls(result["messages"])}


In [ ]:
results = client.evaluate(
    run_agent_trajectory,
    data=dataset_name,
    evaluators=[trajectory_match, extra_steps, missing_steps],
    experiment_prefix=qualify("trajectory"),
    max_concurrency=2,
)
print(f"View at: {results.experiment_name}")


### 3.5 Pairwise evaluation (A/B, LLM-as-judge)

The evals so far score each experiment **absolutely** against a rubric. But when
you're choosing between two candidates — a new prompt vs. the old one, a cheaper
model vs. the incumbent — the question is **relative**: *which response is
better?* Pairwise (A/B) evaluation shows both candidates' answers to a judge
**side by side** and asks it to pick a winner. This sidesteps the calibration
problem where a judge marks everything `True` on an absolute scale but can still
reliably say *A beats B*. Here we tailor the pairwise judge to the **same `correctness` criteria** we scored absolutely in 3.2, so it's a like-for-like comparison of the crew/catering agent's answers.

We use `client.evaluate_comparative(...)`, which takes **two existing
experiments** and a **comparative evaluator**. The evaluator receives the
`outputs` from both experiments for the same example plus the `example` itself,
and returns a ranking. We reuse the final-response experiment from 2.2 as
candidate **A**, and build a deliberately weaker candidate **B** (a terse agent
that returns only its raw response with no grounding narrative) so the judge has
a real preference to express.

**In the UI:** open two experiments in *Datasets & Testing*, select both, and
click *Compare* → *Pairwise* to configure the same judge with no code.


In [ ]:
def target_final_baseline(inputs: dict) -> dict:
    # run_agent_final expects the full inputs dict and returns {response, files}.
    out = run_agent_final(inputs)
    # Weaker candidate: keep only the first sentence of the response, dropping
    # any grounding narrative the judge would reward.
    resp = out["response"] or ""
    terse = resp.split(". ")[0].strip()
    return {"response": terse, "files": out.get("files", {})}

baseline_results = client.evaluate(
    target_final_baseline,
    data=dataset_name,
    evaluators=[correctness_evaluator],
    experiment_prefix=qualify("crew-catering-final-response-baseline"),
    max_concurrency=2,
)
print(f"Baseline experiment: {baseline_results.experiment_name}")

Now the pairwise judge — the **comparative twin of the `correctness` judge** from 3.2, scoring the same crew/catering criteria (accurate, grounded in tool data, compliance risks surfaced) but relatively instead of absolutely. It sees the ops request, the success rubric, and
**both** agents' responses (order randomized to avoid position bias), then
assigns a preference. The comparative evaluator returns `scores` positionally
for the two runs — `1` for the winner, `0` for the loser (`0.5`/`0.5` on a tie)
— which LangSmith aggregates into a win rate.


In [ ]:
import random

class PairwiseGrade(TypedDict):
    """Which response better satisfied the crew/catering request."""
    preferred: int   # 1 if Response A is better, 2 if Response B is better, 0 if tie
    reasoning: str   # one-sentence explanation

pairwise_judge_prompt = """You are comparing two crew & catering operations assistants for an airline.

This is the comparative form of the same `correctness` criteria we judged absolutely above: pick the response that is more correct, better grounded in the tool data, more complete, and clearer.

You'll see the ops request, a RUBRIC describing what success looks like, and TWO candidate responses (A and B).

Prefer the response that:
- reports the requested crew roster / duty-time-compliance / catering figures accurately and grounded in tool output (a grounded answer with the actual figures beats a terse one that omits them);
- surfaces any duty-time or rest compliance risk explicitly rather than glossing over it;
- is complete and clearly communicated for an ops reader.

A response that refuses a legitimate ops request, fabricates roster/meal/duty data, or misses a compliance breach should lose.

Return preferred=1 if A is better, preferred=2 if B is better, or preferred=0 only if they are genuinely equivalent. Give one short sentence of reasoning."""

pairwise_judge = judge_model.with_structured_output(PairwiseGrade)

def pairwise_evaluator(runs: list, inputs: dict, outputs: list, reference_outputs: dict) -> dict:
    # A comparative evaluator must return `scores` as a dict mapping each
    # run's id -> score. `outputs[i]` lines up with `runs[i]`.
    # Shuffle which experiment is shown as A vs. B so the judge can't exploit
    # position, then map the verdict back to the original run ids.
    order = [0, 1]
    random.shuffle(order)
    a_idx, b_idx = order
    grade = pairwise_judge.invoke([
        SystemMessage(content=pairwise_judge_prompt),
        HumanMessage(content=(
            f"Ops request: {inputs.get('query')}\n\n"
            f"Success rubric: {reference_outputs['reference_answer']}\n\n"
            f"Response A: {outputs[a_idx].get('response')}\n\n"
            f"Response B: {outputs[b_idx].get('response')}"
        )),
    ])
    # 0.5/0.5 on a tie; winner gets 1, loser 0. Keyed by run id.
    a_id, b_id = str(runs[a_idx].id), str(runs[b_idx].id)
    scores = {a_id: 0.5, b_id: 0.5}
    if grade["preferred"] == 1:
        scores[a_id], scores[b_id] = 1, 0
    elif grade["preferred"] == 2:
        scores[a_id], scores[b_id] = 0, 1
    return {"key": "pairwise_preference", "scores": scores, "comment": grade["reasoning"]}


### Run the comparison. 
`evaluate_comparative` takes the **two experiment names**
(candidate A from 2.2, the baseline B we just ran) and the comparative
evaluator. The result view shows per-example winners and the overall win rate.


In [ ]:
from langsmith import Client

client = Client()

comparison = client.evaluate(
    [results.experiment_name, baseline_results.experiment_name],
    evaluators=[pairwise_evaluator],
    experiment_prefix=qualify("crew-catering-pairwise"),
    max_concurrency=2,
)
print("Pairwise comparison complete — open the compare view in LangSmith to see win rates.")

## Part 4. Online Evaluations

**Online evals** run automatically against every new trace as it lands in your tracing project — same evaluator as in Part 3, just triggered on incoming runs instead of a dataset.

LangSmith calls these **run rules**. The Python SDK doesn't expose them directly, so we wrap the REST endpoint with a helper at `utils/langsmith_rules.py`. 
Pass in: a project name, an LLM-as-judge prompt, an output schema. Get back: the rule ID and a deep link to inspect it in the UI.

In [ ]:
# Define the LLM-as-judge prompt + schema.
judge_prompt = (
    "You score whether an assistant response satisfied the user's request.\n"
    "Reply with correctness (true/false) and one sentence of comment explaining why."
)

judge_schema = {
    "title": "correctness",
    "description": "Score whether the assistant response was correct/helpful.",
    "type": "object",
    "properties": {
        "correctness": {"type": "boolean", "description": "True if the response was correct/helpful"},
        "comment": {"type": "string", "description": "One short sentence explaining the score"},
    },
    "required": ["correctness", "comment"],
    "strict": True,
}

online_rule = create_run_rule(
    client,
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    display_name=qualify("workshop-online-correctness"),
    sampling_rate=1.0,
    # Score only root traces, not every child LLM/tool/middleware span.
    filter="eq(is_root, true)",
    llm_judge_prompt=judge_prompt,
    llm_judge_schema=judge_schema,
)

print("Rule ID:", online_rule["id"])
print("Open in UI:", online_rule["url"])


## Annotation Queues — Close the Loop

Once runs have **feedback scores** (from the online eval above, or any other source), route the low-scoring ones to a human for review.

LangSmith's annotation queues are that queue. We use **the same `create_run_rule` helper** — this time with `add_to_annotation_queue_id` set instead of an LLM judge. 
Any run matching the filter is added to the queue automatically.


In [ ]:
# The queue is used for two things: reviewing low-`correctness` runs routed by
# the automation rule below, and labeling `briefing_completeness` for Align Evals.
# Give it reviewer instructions + both feedback keys so there's actually
# something to score (a bare queue has neither).
queue = get_or_create_annotation_queue(
    client,
    name=qualify("modular-workshops-needs-review"),
    description="Runs routed here by the workshop's correctness automation rule.",
    rubric_instructions=(
        "Review each crew & catering agent response as a desk expert would.\n"
        "- correctness: 1 if the response satisfies the request (accurate, grounded "
        "in tool data, no fabrication); 0 otherwise.\n"
        "- briefing_completeness: 1 ONLY if the briefing covers all three of crew, "
        "duty-time/rest compliance, AND catering/special meals; 0 if any is missing."
    ),
    rubric_items=[
        {"feedback_key": "correctness",
         "description": "Did the response satisfy the request, grounded in tool data?",
         "is_required": True},
        {"feedback_key": "briefing_completeness",
         "description": "Covers crew AND compliance AND catering? (house-style rule)",
         "is_required": False},
    ],
)
print(f"Queue: {queue.name} (id={queue.id})")


In [ ]:
# Same helper, no evaluator this time -- just a routing rule.
# Filter: only root traces (is_root=true) with correctness > 0.5.
queue_rule = create_run_rule(
    client,
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    display_name=qualify("workshop-route-correctness"),
    sampling_rate=1.0,
    filter=(
        'and('
        'eq(is_root, true), '
        'eq(feedback_key, "correctness"), '
        'gt(feedback_score, 0.5)'
        ')'
    ),
    add_to_annotation_queue_id=queue.id,
)

print("Queue rule ID:", queue_rule["id"])
print("Open in UI:    ", queue_rule["url"])


### Trigger both rules

Both rules are live. Run a few more light traces and you'll see:

1. The online eval fires on each new trace and attaches a `correctness` feedback score (~30s delay).
2. The queue rule fires on each *new feedback* that matches its filter (low correctness) and routes the run to the review queue.


In [ ]:
trigger_prompts = [
    "In one line, who is the purser on flight AC418?",
    "In one sentence, is a 15h duty period compliant for cabin crew?",
    "In one sentence, how many special meals total are on flight AC902?",
]

time.sleep(100)

total = 0.0
for q in trigger_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].text[:120]}")

# Use the tenant_id LangSmith returned with the rule so the link works regardless of workspace.
tenant_id = queue_rule["payload"]["tenant_id"]

print(f"\nTotal: {total:.1f}s.")
print(f"\nOnline eval rule:  {online_rule['url']}")
print(f"Queue rule:        {queue_rule['url']}")
print(f"Queue (review UI): https://smith.langchain.com/o/{tenant_id}/annotation-queues/{queue.id}")
print("\nFeedback shows up in the rule pages within ~30s; queue placements follow once feedback lands.")


### Common run-rule patterns

Swap the `filter` to build different rules:

| Use Case | `filter` |
|---|---|
| Low online correctness | `and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))` |
| Errored runs | `eq(status, "error")` |
| Slow runs | `gt(latency, 10)` |
| Long-running tool calls | `and(eq(run_type, "tool"), gt(latency, 3))` |
| Specific tool fired | `eq(name, "duty_time_check")` |

Both rule types — online eval and queue routing — go through the same `create_run_rule` helper. 
Use `delete_run_rule(client, rule_id)` to tear them down when you're done.


## Align Evals — Tune the Judge to a Human Expert

The online eval and annotation queue above assume the `correctness` judge is *right*. But an LLM-as-judge is only as trustworthy as its agreement with a human expert — and for objective checks like `correctness` the model is already well-calibrated out of the box, so there's little to tune.

The judgments worth aligning are the **subjective, house-style** ones the model has no prior about. For our crew & catering desk that's **briefing completeness**: the desk's standard says a pre-departure briefing must cover **crew _and_ compliance _and_ catering** — a briefing that nails two of the three is *not* "good enough," even though a model left to its own devices will happily call it complete.

**Align Evals** (LangSmith's *Align Evaluator* workflow) closes that gap. You label a handful of real outputs the way an expert would, then iterate the judge prompt in the **Evaluator Playground** until its scores match your labels. The metric is the **alignment score** — the % of examples where the judge agrees with the human. The loop has four steps:

1. **Dataset + experiment** — build a dataset of briefings and run an experiment on it (Align Evaluator labels *experiment runs*, so a dataset alone isn't enough).
2. **Review & label** — send that experiment for human review, score each run, and add it to a reference dataset.
3. **Test the judge prompt** — run it over the labeled set in the Evaluator Playground and read the alignment score.
4. **Refine & repeat** — investigate misaligned examples, tighten the rubric, re-test.

📖 Full guide: [Improve LLM-as-judge evaluators using human feedback →](https://docs.langchain.com/langsmith/improve-judge-evaluator-feedback)


### 1. Build a dataset and run an experiment on it

Align Evaluator aligns a judge against **labeled experiment runs**, so it needs a **dataset with at least one experiment** — a dataset of examples alone has nothing to annotate. So we do two things here:

1. Create a small **briefing dataset** — a spread of prompts, some inviting a full crew + compliance + catering briefing, others nudging toward a partial one, so the labels end up balanced across *complete* (1) and *incomplete* (0).
2. Run an **experiment** on it (just the agent's briefing output — no evaluator needed yet). That experiment is what you'll send for human review in the next step.


In [ ]:
# A spread of briefing requests: some invite a full crew+compliance+catering
# briefing, others nudge toward a partial one -> balanced 1/0 labels to align on.
briefing_examples = [
    {"inputs": {"query": "Give me a full pre-departure briefing for flight AC418."}},
    {"inputs": {"query": "Give me a pre-departure briefing for flight AC902."}},
    {"inputs": {"query": "Full ops briefing for AC418 before departure, please."}},
    {"inputs": {"query": "Quick — just tell me the crew on AC418."}},              # likely partial (crew only)
    {"inputs": {"query": "What special meals are loaded on AC418?"}},               # likely partial (catering only)
    {"inputs": {"query": "Brief me on AC902, focus on duty-time compliance."}},     # likely partial (compliance only)
]

# 1) Create (or refresh) the briefing dataset.
briefing_dataset_name = qualify("crew-briefings")
if client.has_dataset(dataset_name=briefing_dataset_name):
    client.delete_dataset(dataset_id=client.read_dataset(dataset_name=briefing_dataset_name).id)
briefing_dataset = client.create_dataset(briefing_dataset_name)
client.create_examples(
    inputs=[e["inputs"] for e in briefing_examples],
    dataset_id=briefing_dataset.id,
)
print(f"Dataset '{briefing_dataset_name}' ready with {len(briefing_examples)} examples.")

# 2) Run an experiment on the dataset. No evaluator -- we only need the runs so a
#    human can label them. `run_agent_final` (from Part 3) returns the briefing.
briefing_experiment = client.evaluate(
    run_agent_final,
    data=briefing_dataset_name,
    experiment_prefix=qualify("crew-briefings"),
    max_concurrency=2,
)
print(f"Experiment to review: {briefing_experiment.experiment_name}")
print("Open Datasets & Experiments -> this experiment to send its runs for review.")


### 2. Send the experiment for review, then label it

Now hand those experiment runs to a human. In **Datasets & Experiments → your `crew-briefings` dataset → Evaluators tab**, click **+ Evaluator → Create from labeled data**, name the feedback key **`briefing_completeness`**, then **select the experiment you just ran**. LangSmith adds its runs to an annotation queue owned by this evaluator.

> Why this order: Align Evaluator sources labels from **experiment runs**, not from raw dataset examples — that's why we ran an experiment first, and why creating an empty dataset by hand leaves nothing to annotate.

In that queue, score each briefing the way the **desk expert** would — not the way a lenient model would:

- Score **1** only if the briefing covers **all three**: crew, duty-time/rest compliance, *and* catering/special meals.
- Score **0** if any of the three is missing — even a well-written crew-only or catering-only answer is incomplete by house style.

After scoring each run, click **Add to Reference Dataset** — this is the step that turns your labels into the reference set the Playground aligns against. Aim for a **balance of 0s and 1s** (run the experiment cell again for more runs if needed). Diversity matters more than volume.


### 3. Test the judge prompt in the Evaluator Playground

With runs labeled and added to the reference dataset, open the **Evaluator Playground** (from the evaluator's page, click **Evaluator Playground**), paste the starter judge prompt below, enable **reasoning**, and click **Start Alignment**.

LangSmith runs the judge over your labeled set and shows the **alignment score** (% agreement with your labels), plus which examples it got wrong.

**Starter `briefing_completeness` judge prompt** (paste into the playground):

```text
You are grading a pre-departure briefing from an airline crew & catering desk.

House-style rule: a briefing is COMPLETE (score 1) only if it covers ALL THREE of:
  1. Crew — who is on duty for the flight.
  2. Compliance — duty-time / rest status or an explicit flag that it must be checked.
  3. Catering — special-meal / provisioning details.

If ANY of the three is missing, the briefing is INCOMPLETE (score 0) — even if what
is present is accurate and well written. A polished crew-only or catering-only answer
still scores 0.

Return score=1 (complete) or score=0 (incomplete) and one sentence naming which of the
three areas are present and which are missing.
```

You'll likely see the judge start too lenient — scoring partial briefings as complete — which is exactly the gap alignment closes.


### 4. Refine & repeat until the judge agrees with you

Iterate in the playground; save the prompt whenever the alignment score improves (updates aren't saved automatically — the playground compares against your last *saved* version).

Tactics that move the score:

- **Investigate misaligned examples** — hover the LLM score to read its **reasoning**; group the misses into failure modes (e.g. "counts catering as covered when only meal *codes* are listed") and add an explicit instruction for each.
- **Pin down the house rule** — spell out edge cases the model gets wrong (e.g. "a compliance line that says 'cannot verify' still counts as covering compliance").
- **Add more labels, then re-check** — expand the reference set so you don't overfit the first handful.

When the alignment score is high enough to trust, promote this `briefing_completeness` judge into an **online evaluator** (same `create_run_rule` pattern from Part 4) — now you have a *domain* quality signal, aligned to your desk's standard, scoring every new briefing automatically.


## Recap

| Part | What | API |
|---|---|---|
| **1. Prompt engineering** | Author, version, and share prompts; pull them into code | `client.push_prompt(...)` / `client.pull_prompt(...)` |
| **Warm-up** | Generate a few traces with the crew & catering agent | `agent.invoke(...)` |
| **2. Tracing + querying** | Auto-capture every run; pull back by filter | `LANGSMITH_TRACING=true`, `client.list_runs(filter=...)` |
| **3. Offline evals** | Score on demand against a dataset | `model.with_structured_output(...)` + `client.evaluate` |
| **4. Online evals** | Score every new trace automatically | `create_run_rule(..., llm_judge_prompt=..., llm_judge_schema=...)` |
| **Annotation queues** | Route flagged runs for human review | `create_run_rule(..., add_to_annotation_queue_id=...)` |
| **Align Evals** | Tune a subjective judge to a human expert (alignment score) | label in queue → **Evaluator Playground** → refine |

The full loop: trace → online eval scores it → run rule routes low scores to the queue → human reviews → **Align Evals tunes the judge to match that human** → fixes flow into the next dataset.

**Next:** Module 5 — **Engine** automates this entire loop (detect → diagnose → PR → evaluator) on your deployed agent.